## Song Analysis

In [2]:
%run prepare.ipynb

### Top Tracks

In [49]:
top_tracks.head(10).reset_index()

,track,date,ms_played,uri,h
0,Nettles,42,43035338,111,12.0
1,Hampstead,39,11380568,53,3.2
2,Some Protector,41,11296063,53,3.1
3,Best Guess,38,10452604,48,2.9
4,What Was That,39,10362873,61,2.9
5,twilight zone,40,10165859,58,2.8
6,dandelion,39,9753788,55,2.7
7,Greedy,38,9554228,48,2.7
8,Angels All Around Me…,29,8799354,37,2.4
9,Headphones On,36,8669441,43,2.4


### Top Tracks by Month

In [50]:
monthly_top_tracks[[col for col in monthly_top_tracks if col.year == int(YEAR)]].head(5)

,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
1,i wish i hated you,Greedy,Best Guess,twilight zone,Sue me,Nettles,Nettles,The Subway,Cinnamon Bread,The Fate of Ophelia,Wonderful,March of the Witch Hunters
2,true story,"Sticky (feat. GloRilla, Sexyy Red & Lil Wayne)",Dandelion,Fuzzy Feeling,Angels All Around Me…,Everyday Is A Winding Road,Man Of The Year,American Teenager,Hunter,Elizabeth Taylor,No Good Deed,Wonderful
3,eternal sunshine,Nomad,past life,dandelion,dandelion,Time To Say Goodbye,What Was That,Big Deal,Bruises Off The Peach,The Life of a Showgirl (feat. Sabrina Carpenter),Sun Bleached Flies,No Good Deed
4,bye,Busy Woman,dandelion,"Sally, When The Wine Runs Out",What Was That,Man Of The Year,Shapeshifter,Some Protector,Multiple Endings,Opalite,Bruises Off The Peach - Live Performance,Every Day More Wicked
5,supernatural,intro (end of the world),drive ME crazy!,Sunshine & Rain...,twilight zone,Money is Everything,Favourite Daughter,Somebody Else - Spotify Singles,Andromeda,Actually Romantic,"Big Guy - from ""The SpongeBob Movie: Search fo...",As Long As You’re Mine


### Top Single-day Streams

In [7]:
single_day_streams = dated_streams[dated_streams["reason_end"] == "trackdone"].copy()
single_day_streams["date"] = single_day_streams.index.date
single_day_streams = single_day_streams.pivot_table(
    index=['artist','track', 'date'],
    values=['uri'],
    aggfunc={'uri': 'count'}
).sort_values(by=['track', 'date']).reset_index()

single_day_streams = single_day_streams[single_day_streams["uri"] >= 5]
single_day_streams.sort_values(by='uri', ascending=False)

,artist,track,date,uri
1909,Lorde,Man Of The Year,2025-05-29,9
597,grentperez,Cherry Wine,2025-04-14,7
1622,Lorde,If She Could See Me Now,2025-06-29,6
2145,Ethel Cain,Nettles,2025-06-23,6
2148,Ethel Cain,Nettles,2025-06-26,6
766,Lorde,David,2025-06-29,5
1290,Ariana Grande,Greedy,2025-02-12,5
2140,Ethel Cain,Nettles,2025-06-16,5
2979,Ethel Cain,Sun Bleached Flies,2025-11-04,5


### Consecutive Streaming Days

In [52]:
dated_streams['date'] = pd.to_datetime(dated_streams['date'])
daily_streams = dated_streams.pivot_table(
    index=['artist','track', 'date'],
    values=['uri'],
    aggfunc={'uri': 'count'}
).sort_values(by=['track', 'date']).reset_index()

streaks = {}
for track, date in zip(daily_streams["track"], daily_streams["date"]):
  if (track not in streaks): streaks[track] = {}

  yesterday = date-pd.Timedelta(days=1)
  if (yesterday in streaks[track]):
    streaks[track][date] = streaks[track][yesterday] + 1
  else:
    streaks[track][date] = 1

reformed_data = [
    [outer_key, inner_key, value]
    for outer_key, inner_dict in streaks.items()
    for inner_key, value in inner_dict.items()
]

streaks = pd.DataFrame(reformed_data,columns=["track", "date", "streak"]).sort_values(by="streak", ascending=False)
streaks = streaks[streaks['streak'] > 5] 

streaks['end'] = pd.to_datetime(streaks['date'])
streaks['start'] = streaks['end'] - pd.to_timedelta(streaks['streak'] - 1, unit='d') #+ pd.Timedelta(days=1)
streaks[['track', 'streak', 'start', 'end']]

,track,streak,start,end
3733,Some Protector,7,2025-05-24,2025-05-30
2870,Nettles,7,2025-06-20,2025-06-26
867,Clearblue,6,2025-07-12,2025-07-17
967,Current Affairs,6,2025-07-12,2025-07-17
3732,Some Protector,6,2025-05-24,2025-05-29
1408,Favourite Daughter,6,2025-06-27,2025-07-02
2869,Nettles,6,2025-06-20,2025-06-25
2593,Man Of The Year,6,2025-06-26,2025-07-01
1415,Favourite Daughter,6,2025-07-12,2025-07-17
2601,Man Of The Year,6,2025-07-12,2025-07-17
